# Kuna-Chen, Chen ML  Dimensionality Reduction_train

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import seaborn as sns


from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV,  KFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import accuracy_score, mean_squared_error, roc_curve, auc, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE, Isomap

from tensorflow.keras.datasets import cifar10
from joblib import dump
import umap

## Step 1. Choose One of the Following Public Datasets

In [ ]:
# Load data
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# CIFAR-10 has 10 classes
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def visualize_cifar10_samples(X, y, class_names, samples_per_class,random_seed = 2025):
    """
    Displays a grid of sample images for each class in CIFAR-10.
    X: image array, shape (N, 32, 32, 3)
    y: labels array, shape (N, 1)
    class_names: list of 10 class names
    samples_per_class: how many examples to show per class
    """
    np.random.seed(random_seed)
    y = y.squeeze()  # Convert from shape (N,1) to (N,) if needed
    num_classes = len(class_names)
    plt.figure(figsize=(samples_per_class * 2, num_classes * 2))
    for cls_idx in range(num_classes):
        idxs = np.flatnonzero(y == cls_idx)
        idxs = np.random.choice(idxs, samples_per_class, replace=False)
        for i, idx in enumerate(idxs):
            plt.subplot(num_classes, samples_per_class, cls_idx * samples_per_class + i + 1)
            plt.imshow(X[idx])
            plt.axis('off')
            if i == 0:
                plt.title(class_names[cls_idx])
    plt.tight_layout()
    plt.show()

# Example usage:
visualize_cifar10_samples(X_train, y_train, class_names,5)


## Step 2. Data Preparation and Initial Visualization

### Task1 : Provide a summary of findings from the exploratory data analysis (bullet points or short analysis).

In [ ]:
X_train.shape, X_test.shape,y_train.shape,y_test.shape

In [ ]:
y_train_flat = y_train.squeeze()
class_counts = np.bincount(y_train_flat)
for i, count in enumerate(class_counts):
    print(f"{class_names[i]}: {count} images")

### Task2 : Provide at least three visualizations showing trends or insights from the dataset.

I plot the image in to RGB and by different Gray Scale

In [ ]:
def visualize_cifar10_channels(X, y, class_names, samples_per_class, channel=None, random_seed=2025):
    """
    Displays a grid of sample images for each class in CIFAR-10.
    
    X: Image array, shape (N, 32, 32, 3)
    y: Label array, shape (N, 1) or (N,)
    class_names: List of 10 class names
    samples_per_class: Number of samples to show per class
    channel: None for full RGB image, 0=R, 1=G, 2=B for single channel
    """
    y = y.squeeze()
    np.random.seed(random_seed)
    num_classes = len(class_names)
    plt.figure(figsize=(samples_per_class * 2, num_classes * 2))

    for cls_idx in range(num_classes):
        idxs = np.flatnonzero(y == cls_idx)
        idxs = np.random.choice(idxs, samples_per_class, replace=False)
        
        for i, idx in enumerate(idxs):
            plt.subplot(num_classes, samples_per_class, cls_idx * samples_per_class + i + 1)
            
            if channel is None:
                plt.imshow(X[idx])
            else:
                img = np.zeros_like(X[idx])
                img[:,:,channel] = X[idx][:,:,channel]
                plt.imshow(img)
            
            plt.axis('off')
            if i == 0:
                plt.title(class_names[cls_idx])
    
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_cifar10_samples(X_train, y_train, "a",3)
for i in range(3):
    visualize_cifar10_samples(X_train[:,:,:,i], y_train, "a",3)
for i in range(3):
    visualize_cifar10_channels(X_train, y_train, "a", 3, channel=i)

In [ ]:
for i, class_name in enumerate(class_names):
    class_indices = np.where(y_train_flat == i)[0]
    class_images = X_train[class_indices]
    
    class_means = np.mean(class_images, axis=(0, 1, 2))
    class_stds = np.std(class_images, axis=(0, 1, 2))
    
    print(f"\nClass: {class_name}")
    for j, channel in enumerate(['Red', 'Green', 'Blue']):
        print(f"{channel}: Mean={class_means[j]:.2f}, Std={class_stds[j]:.2f}")

### Task3 : Provide a written summary of the preprocessing steps.

I verified the class distribution was balanced with 5000 images per class. For visual exploration, I created functions to display samples in both full RGB and separated grayscale representations of individual color channels. I also calculated mean and standard deviation values for each color channel across all classes to establish baseline statistical characteristics before model training.


## Step 3: Initial Classification (No Dimensionality Reduction)

In [ ]:
reshape_transformer = FunctionTransformer(lambda X: X.reshape(X.shape[0], -1))

Xtrans = Pipeline([
    ('reshape', reshape_transformer),  
    ('scaler', StandardScaler()),      
])

Ytrans = Pipeline([
    ('onehot', OneHotEncoder(sparse_output=False)) 
])

In [ ]:
X_train_transformed = Xtrans.fit_transform(X_train)
y_train_transformed = Ytrans.fit_transform(y_train)


print("X_train_transformed.shape:", X_train_transformed.shape)  
print("y_train_transformed.shape:", y_train_transformed.shape)

### a. Support Vector Classifier (SVC)

In [ ]:
X_train_model = []
y_train_model = []

y_train_flat = y_train.ravel() 

np.random.seed(2025) 

for class_id in range(10):  
    class_indices = np.where(y_train_flat == class_id)[0]
    selected_indices = np.random.choice(class_indices, size=500, replace=False)
    X_train_model.append(X_train[selected_indices])
    y_train_model.append(y_train[selected_indices])

X_train_model = np.vstack(X_train_model)
y_train_model = np.vstack(y_train_model)

In [ ]:
visualize_cifar10_samples(X_train_model, y_train_model, class_names ,3)

In [ ]:
SVC_pipe = Pipeline([
    ("Xtrans", Xtrans),  
    ("svc", SVC())
])

In [ ]:
param_grid_svc = {
    'svc__kernel': ["rbf", "linear"],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ["scale", "auto"]
}

In [ ]:
grid_search_svc = GridSearchCV(
    SVC_pipe,
    param_grid=param_grid_svc,
    scoring='accuracy',  
    refit=True,
    verbose=4
)

In [ ]:
start_time_svc = time.time()
grid_search_svc.fit(X_train_model, y_train_model.ravel())
svc_training_time = time.time() - start_time_svc

In [ ]:
best_params_svc = grid_search_svc.best_params_
svc_cv_accuracy = grid_search_svc.best_score_
best_model_svc = grid_search_svc.best_estimator_

In [ ]:
print(best_params_svc,svc_training_time,svc_cv_accuracy)

In [ ]:
SVC_best_pipe = Pipeline([
    ("Xtrans", Xtrans),  
    ("svc", best_model_svc)
])
dump(SVC_best_pipe, 'model_svc.joblib')

### b. Random Forest

In [ ]:
RanF_pipe = Pipeline([
    ("Xtrans", Xtrans),  
    ("RanF", RandomForestClassifier())
])

In [ ]:
param_grid_RanF = {
    'RanF__n_estimators': [50, 100, 200],
    'RanF__max_depth': [None, 5, 10]
}

In [ ]:
grid_search_RanF = GridSearchCV(
    RanF_pipe,
    param_grid=param_grid_RanF,
    scoring='accuracy',  
    refit=True,
    verbose=2
)

In [ ]:
start_time_RanF = time.time()
grid_search_RanF.fit(X_train_model, y_train_model.ravel())
rf_training_time = time.time() - start_time_RanF

In [ ]:
best_params_RanF = grid_search_RanF.best_params_
RanF_cv_accuracy = grid_search_RanF.best_score_
best_model_RanF = grid_search_RanF.best_estimator_

In [ ]:
print(best_params_RanF,rf_training_time,RanF_cv_accuracy)
best_model_RanF
RanF_best_pipe = Pipeline([
    ("Xtrans", Xtrans),  
    ("RanF", best_model_RanF)
])
dump(RanF_best_pipe, 'model_RanF.joblib')

### c. Logistic Regression (multi-class)

In [ ]:
LogR_pipe = Pipeline([
    ("Xtrans", Xtrans),
    ("LogR", LogisticRegression())
])


In [ ]:
param_grid_LogR = {
    "LogR__penalty": ["l2"],
    "LogR__solver": ["saga", "lbfgs"],
    "LogR__C": [ 0.1, 1, 10],
    "LogR__max_iter": [500,1000,2000]
}

In [ ]:
grid_search_LogR = GridSearchCV(
    LogR_pipe,
    param_grid=param_grid_LogR,
    scoring='accuracy',
    refit=True,
    verbose=2,
    n_jobs=-1  # Use all available processors
)

In [ ]:
start_time_LogR = time.time()
grid_search_LogR.fit(X_train_model, y_train_model.ravel())
LogR_training_time = time.time() - start_time_LogR

In [ ]:
best_params_LogR = grid_search_LogR.best_params_
LogR_cv_accuracy = grid_search_LogR.best_score_
best_model_LogR = grid_search_LogR.best_estimator_

In [ ]:
print(best_params_LogR,LogR_training_time,LogR_cv_accuracy)
best_model_LogR
LogR_best_pipe = Pipeline([
    ("Xtrans", Xtrans),
    ("LogR", best_model_LogR)
])
dump(LogR_best_pipe, 'model_LogR.joblib')

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['SVC', 'RandomForest', 'LogisticRegression'],
    'accuracy': [svc_cv_accuracy, RanF_cv_accuracy, LogR_cv_accuracy],  # Fixed variable names
    'training_time': [svc_training_time, rf_training_time, LogR_training_time],
})

In [ ]:
comparison_df

## Step 4: PCA Dimensionality Reduction

### part1

In [ ]:
X_test_model = []
y_test_model = []

y_test_flat = y_test.ravel() 

np.random.seed(2025) 

for class_id in range(10):  
    class_indices = np.where(y_test_flat == class_id)[0]
    # Take up to 100 samples per class (or all if less than 100)
    selected_indices = np.random.choice(class_indices, size=min(100, len(class_indices)), replace=False)
    X_test_model.append(X_test[selected_indices])
    y_test_model.append(y_test[selected_indices])

X_test_model = np.vstack(X_test_model)
y_test_model = np.vstack(y_test_model)

In [ ]:
X_train_flat = X_train_model.reshape(X_train_model.shape[0], -1)  # Flatten the images
X_test_flat = X_test_model.reshape(X_test_model.shape[0], -1)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_flat_scaled = scaler.fit_transform(X_train_flat)
X_test_flat_scaled = scaler.transform(X_test_flat)

In [ ]:
pca = PCA()
pca.fit(X_train_flat_scaled)

# Plot explained variance ratio
plt.figure(figsize=(10, 6))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance by Components')
plt.axhline(y=0.9, color='r', linestyle='--', label='90% Variance Threshold')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = np.argmax(cumulative_variance >= 0.9) + 1
print(f"Number of components needed for 90% variance: {n_components_90}")

In [ ]:
def calculate_rmse(original, reconstructed):
    return np.sqrt(mean_squared_error(original.flatten(), reconstructed.flatten()))

In [ ]:
def visualize_reconstruction(original, reconstructed, titles=None):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    axes[0].imshow(original)
    axes[0].set_title(titles[0] if titles else "Original")
    axes[0].axis('off')
    
    axes[1].imshow(reconstructed.clip(0, 255).astype('uint8'))
    axes[1].set_title(titles[1] if titles else "Reconstructed")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
component_numbers = [10, 50, 100, 200, 300, 500]
rmse_values = []

# Select a few sample images
sample_indices = np.random.choice(len(X_train_model), 3, replace=False)

In [ ]:
for n_components in component_numbers:
    # Fit PCA with specific number of components
    pca_n = PCA(n_components=n_components)
    pca_n.fit(X_train_flat_scaled)
    
    # Transform and inverse transform to get reconstructions
    transformed = pca_n.transform(X_train_flat_scaled)
    reconstructed_flat = pca_n.inverse_transform(transformed)
    
    # Reshape back to image format and apply inverse scaling
    reconstructed_scaled = scaler.inverse_transform(reconstructed_flat)
    reconstructed = reconstructed_scaled.reshape(X_train_model.shape)
    
    # Calculate average RMSE across all images
    rmse = calculate_rmse(X_train_flat, reconstructed_scaled)
    rmse_values.append(rmse)
    print(f"Components: {n_components}, Average RMSE: {rmse:.4f}")
    
    # Visualize reconstructions for sample images
    for i in sample_indices:
        visualize_reconstruction(
            X_train_model[i], 
            reconstructed[i],
            [f"Original", f"Reconstructed ({n_components} components)"]
        )

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(component_numbers, rmse_values, marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Average RMSE')
plt.title('Reconstruction Error vs. Number of Components')
plt.grid(True)
plt.show()

### part2 PCA&SVC

In [ ]:
pca_svc_pipe = Pipeline([
    ("reshape", reshape_transformer),
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("svc", SVC())
])

In [ ]:
param_grid_pca_svc = {
    'pca__n_components': [50, 100],
    'svc__kernel': ["rbf", "linear"],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ["scale", "auto"]
}


In [ ]:
grid_search_pca_svc = GridSearchCV(
    pca_svc_pipe,
    param_grid=param_grid_pca_svc,
    scoring='accuracy',
    refit=True,
    verbose=2
)


In [ ]:
start_time_pca_svc = time.time()
grid_search_pca_svc.fit(X_train_model, y_train_model.ravel())
pca_svc_training_time = time.time() - start_time_pca_svc

In [ ]:
best_params_pca_svc = grid_search_pca_svc.best_params_
pca_rf_cv_accuracy = grid_search_pca_svc.best_score_
best_model_pca_svc = grid_search_pca_svc.best_estimator_

dump(best_model_pca_svc, 'model_pca_svc.joblib')

In [ ]:
print("PCA + SVC Results:")
print(f"Best parameters: {best_params_pca_svc}")
print(f"Training time: {pca_svc_training_time:.2f} seconds")
print(f"CV accuracy: {pca_rf_cv_accuracy:.4f}")


### part3 PCA&Random Forest

In [ ]:
pca_rf_pipe = Pipeline([
    ("reshape", reshape_transformer),
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("rf", RandomForestClassifier())
])

In [ ]:
param_grid_pca_rf = {
    'pca__n_components': [50, 100],
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [None, 5, 10]
}

In [ ]:
grid_search_pca_rf = GridSearchCV(
    pca_rf_pipe,
    param_grid=param_grid_pca_rf,
    scoring='accuracy',
    refit=True,
    verbose=2
)

In [ ]:
start_time_pca_rf = time.time()
grid_search_pca_rf.fit(X_train_model, y_train_model.ravel())
pca_rf_training_time = time.time() - start_time_pca_rf

In [ ]:
best_params_pca_rf = grid_search_pca_rf.best_params_
pca_rf_cv_accuracy = grid_search_pca_rf.best_score_
best_model_pca_rf = grid_search_pca_rf.best_estimator_

dump(best_model_pca_rf, 'model_pca_RanF.joblib')

In [ ]:
print("\nPCA + Random Forest Results:")
print(f"Best parameters: {best_params_pca_rf}")
print(f"Training time: {pca_rf_training_time:.2f} seconds")
print(f"CV accuracy: {pca_rf_cv_accuracy:.4f}")

### Part4 PCA&Logistic Regression

In [ ]:
pca_logreg_pipe = Pipeline([
    ("reshape", reshape_transformer),
    ("scaler", StandardScaler()),
    ("pca", PCA()),
    ("logreg", LogisticRegression())
])

In [ ]:
param_grid_pca_logreg = {
    'pca__n_components': [50, 100],
    'logreg__penalty': ["l2"],
    'logreg__solver': ["saga", "lbfgs"],
    'logreg__C': [0.01, 0.1, 1, 10],
    'logreg__max_iter': [200, 500]
}

In [ ]:

grid_search_pca_logreg = GridSearchCV(
    pca_logreg_pipe,
    param_grid=param_grid_pca_logreg,
    scoring='accuracy',
    refit=True,
    verbose=2
)

In [ ]:
start_time_pca_logreg = time.time()
grid_search_pca_logreg.fit(X_train_model, y_train_model.ravel())
pca_logreg_training_time = time.time() - start_time_pca_logreg

In [ ]:
best_params_pca_logreg = grid_search_pca_logreg.best_params_
pca_logreg_cv_accuracy = grid_search_pca_logreg.best_score_
best_model_pca_logreg = grid_search_pca_logreg.best_estimator_
dump(best_model_pca_logreg, 'model_pca_LogR.joblib')

In [ ]:
print("\nPCA + Logistic Regression Results:")
print(f"Best parameters: {best_params_pca_logreg}")
print(f"Training time: {pca_logreg_training_time:.2f} seconds")
print(f"CV accuracy: {pca_logreg_cv_accuracy:.4f}")


In [ ]:
y_pred_pca_svc = best_model_pca_svc.predict(X_test_model)
pca_svc_test_acc = accuracy_score(y_test_model.ravel(), y_pred_pca_svc)
pca_svc_f1 = f1_score(y_test_model.ravel(), y_pred_pca_svc, average='macro')

y_pred_pca_rf = best_model_pca_rf.predict(X_test_model)
pca_rf_test_acc = accuracy_score(y_test_model.ravel(), y_pred_pca_rf)
pca_rf_f1 = f1_score(y_test_model.ravel(), y_pred_pca_rf, average='macro')

y_pred_pca_logreg = best_model_pca_logreg.predict(X_test_model)
pca_logreg_test_acc = accuracy_score(y_test_model.ravel(), y_pred_pca_logreg)
pca_logreg_f1 = f1_score(y_test_model.ravel(), y_pred_pca_logreg, average='macro')

print("\nTest Set Performance:")
print(f"PCA + SVC: Accuracy = {pca_svc_test_acc:.4f}, F1 = {pca_svc_f1:.4f}")
print(f"PCA + RF: Accuracy = {pca_rf_test_acc:.4f}, F1 = {pca_rf_f1:.4f}")
print(f"PCA + LogReg: Accuracy = {pca_logreg_test_acc:.4f}, F1 = {pca_logreg_f1:.4f}")


## Step 5.manifold learning implementation

In [ ]:
def plot_components(X_transformed, y, title, class_names=None):

    y_flat = y.ravel()
    plt.figure(figsize=(12, 10))
    
    # Scatter plot colored by class
    scatter = plt.scatter(X_transformed[:, 0], X_transformed[:, 1], 
               c=y_flat, alpha=0.8, cmap='viridis')
    
    # Add colorbar and title
    plt.colorbar(scatter, label='Class')
    plt.title(title, fontsize=15)
    plt.xlabel('Component 1', fontsize=12)
    plt.ylabel('Component 2', fontsize=12)
    
    # Add legend with class names if provided
    if class_names:
        handles, labels = scatter.legend_elements()
        legend = plt.legend(handles, class_names, loc='best', 
                          title='Classes', fontsize=10)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    return plt


In [ ]:
def record_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test.ravel(), y_pred)
    f1 = f1_score(y_test.ravel(), y_pred, average='macro')
    cm = confusion_matrix(y_test.ravel(), y_pred)
    return {'accuracy': acc, 'f1_score': f1, 'confusion_matrix': cm}

## part1. t-SNE

In [ ]:
tsne = TSNE(n_components=2, random_state=2025)
X_flat = X_train_model.reshape(X_train_model.shape[0], -1)
X_scaled = StandardScaler().fit_transform(X_flat)
X_tsne = tsne.fit_transform(X_scaled)
# Plot t-SNE embeddings
plot_components(X_tsne, y_train_model, 'CIFAR-10: t-SNE Visualization', class_names)
plt.savefig('tsne_visualization.png', dpi=300)
plt.show()

In [ ]:
tsne_svc_pipe = Pipeline([
    ("reshape", reshape_transformer),
    ("scaler", StandardScaler()),
    ("tsne", TSNE(n_components=50, random_state=2025)),
    ("svc", SVC())
])

In [ ]:
param_grid_tsne_svc = {
    'tsne__perplexity': [30, 50],
    'svc__kernel': ["rbf", "linear"],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ["scale", "auto"]
}

In [ ]:
grid_search_tsne_svc = GridSearchCV(
    tsne_svc_pipe,
    param_grid=param_grid_tsne_svc,
    scoring='accuracy',
    refit=True,
    verbose=2
)

In [ ]:
start_time_tsne_svc = time.time()
grid_search_tsne_svc.fit(X_train_model, y_train_model.ravel())
tsne_svc_training_time = time.time() - start_time_tsne_svc

In [ ]:
best_params_tsne_svc = grid_search_tsne_svc.best_params_
tsne_svc_cv_accuracy = grid_search_tsne_svc.best_score_
best_model_tsne_svc = grid_search_tsne_svc.best_estimator_
dump(best_model_tsne_svc, 'model_tsne_svc.joblib')

In [ ]:
print("t-SNE + SVC Results:")
print(f"Best parameters: {best_params_tsne_svc}")
print(f"Training time: {tsne_svc_training_time:.2f} seconds")
print(f"CV accuracy: {tsne_svc_cv_accuracy:.4f}")


## part2. UMAP

In [ ]:
umap_reducer = umap.UMAP(n_components=2, random_state=2025)
X_flat = X_train_model.reshape(X_train_model.shape[0], -1)
X_scaled = StandardScaler().fit_transform(X_flat)
X_umap = umap_reducer.fit_transform(X_scaled)

# Plot UMAP embeddings
plot_components(X_umap, y_train_model, 'CIFAR-10: UMAP Visualization', class_names)
plt.savefig('umap_visualization.png', dpi=300)
plt.show()

In [ ]:
class UMAPTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=2, n_neighbors=15, min_dist=0.1, random_state=None):
        self.n_components = n_components
        self.n_neighbors = n_neighbors
        self.min_dist = min_dist
        self.random_state = random_state
        
    def fit(self, X, y=None):
        self.umap_ = umap.UMAP(
            n_components=self.n_components,
            n_neighbors=self.n_neighbors,
            min_dist=self.min_dist,
            random_state=self.random_state
        )
        self.umap_.fit(X)
        return self
    
    def transform(self, X):
        return self.umap_.transform(X)

In [ ]:
umap_svc_pipe = Pipeline([
    ("reshape", reshape_transformer),
    ("scaler", StandardScaler()),
    ("umap", UMAPTransformer(n_components=50, random_state=2025)),
    ("svc", SVC())
])

In [ ]:
param_grid_umap_svc = {
    'umap__n_neighbors': [15, 30],
    'umap__min_dist': [0.1, 0.5],
    'svc__kernel': ["rbf", "linear"],
    'svc__C': [0.1, 1, 10],
    'svc__gamma': ["scale", "auto"]
}

In [ ]:
grid_search_umap_svc = GridSearchCV(
    umap_svc_pipe,
    param_grid=param_grid_umap_svc,
    scoring='accuracy',
    refit=True,
    verbose=2
)

In [ ]:
start_time_umap_svc = time.time()
grid_search_umap_svc.fit(X_train_model, y_train_model.ravel())
umap_svc_training_time = time.time() - start_time_umap_svc

In [ ]:
best_params_umap_svc = grid_search_umap_svc.best_params_
umap_svc_cv_accuracy = grid_search_umap_svc.best_score_
best_model_umap_svc = grid_search_umap_svc.best_estimator_
dump(best_model_umap_svc, 'model_umap_svc.joblib')

In [ ]:
print("UMAP + SVC Results:")
print(f"Best parameters: {best_params_umap_svc}")
print(f"Training time: {umap_svc_training_time:.2f} seconds")
print(f"CV accuracy: {umap_svc_cv_accuracy:.4f}")